# 08 – Exploration

Checks whether the features separate hexagons with and without bubble tea shops,
how features relate to each other, and whether shops cluster in space.

**Key findings**
- **Food & drink places within 500 m** is the strongest signal: the median is 26×
  higher in hexagons with a shop.
- Hexagons with shops have about 2× the population density and high schools nearby,
  1.6× more young adults within ~1 km, and are about half as far from a station.
- Median household income is slightly *lower* (0.89×) where shops exist. Bubble tea
  shops favour dense, busy, younger areas over the wealthiest ones.

**Feature selection**
- `catchment_pop` and `catchment_15_34` have a Spearman correlation of 0.98, so
  `catchment_pop` is dropped to avoid duplicate information.
- 10 hexagons are missing income (privacy suppression). LightGBM handles missing
  values natively, so they are left as-is.

**Spatial autocorrelation (Moran's I, queen contiguity, 999 permutations)**
| Variable | Moran's I | p-value |
|---|---|---|
| has_bubble_tea | 0.137 | 0.001 |
| catchment_15_34 | 0.866 | 0.001 |
| food_places_500m | 0.590 | 0.001 |

Shops and especially the features cluster strongly. A random train/test split would
place near-identical neighbours in both sets and overstate accuracy, so the model is
evaluated with **spatial cross-validation** instead.

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

gdf = gpd.read_file("../data/processed/model_features.gpkg")

FEATURES = [
    "dist_station_m", "dist_campus_m", "high_schools_1km", "food_places_500m",
    "catchment_pop", "catchment_15_34", "median_hh_income", "share_15_34", "pop_density",
]

print("Hexagons:", len(gdf))
gdf[FEATURES].isna().sum()

Hexagons: 846


dist_station_m       0
dist_campus_m        0
high_schools_1km     0
food_places_500m     0
catchment_pop        0
catchment_15_34      0
median_hh_income    10
share_15_34          0
pop_density          0
dtype: int64

In [2]:
comp = gdf.groupby("has_bubble_tea")[FEATURES].median().T
comp.columns = ["no_shop", "has_shop"]
comp["ratio"] = comp["has_shop"] / comp["no_shop"]
comp.round(2)

,no_shop,has_shop,ratio
dist_station_m,2214.15,1227.05,0.55
dist_campus_m,4501.19,3335.27,0.74
high_schools_1km,1.00,2.00,2.00
food_places_500m,1.00,26.00,26.00
catchment_pop,19312.04,28462.28,1.47
catchment_15_34,5039.72,7952.49,1.58
median_hh_income,94929.68,84474.26,0.89
share_15_34,0.26,0.30,1.15
pop_density,3186.26,6685.92,2.10


In [3]:
gdf[FEATURES].corr(method="spearman").round(2)

,dist_station_m,dist_campus_m,high_schools_1km,food_places_500m,catchment_pop,catchment_15_34,median_hh_income,share_15_34,pop_density
dist_station_m,1.00,-0.06,-0.31,-0.35,-0.54,-0.53,0.13,-0.09,-0.42
dist_campus_m,-0.06,1.00,-0.05,-0.14,-0.04,-0.12,0.11,-0.34,-0.06
high_schools_1km,-0.31,-0.05,1.00,0.34,0.51,0.50,-0.17,0.10,0.47
food_places_500m,-0.35,-0.14,0.34,1.00,0.43,0.43,-0.31,0.18,0.50
catchment_pop,-0.54,-0.04,0.51,0.43,1.00,0.98,-0.29,0.12,0.83
catchment_15_34,-0.53,-0.12,0.50,0.43,0.98,1.00,-0.34,0.24,0.81
median_hh_income,0.13,0.11,-0.17,-0.31,-0.29,-0.34,1.00,-0.41,-0.39
share_15_34,-0.09,-0.34,0.10,0.18,0.12,0.24,-0.41,1.00,0.14
pop_density,-0.42,-0.06,0.47,0.50,0.83,0.81,-0.39,0.14,1.00


In [4]:
from libpysal.weights import Queen
from esda.moran import Moran

# Neighbours = hexagons that share an edge
w = Queen.from_dataframe(gdf, use_index=False)
w.transform = "r"
print("Hexagons with no neighbours:", len(w.islands))

for col in ["has_bubble_tea", "catchment_15_34", "food_places_500m"]:
    m = Moran(gdf[col].values, w, permutations=999)
    print(f"{col:20s}  Moran's I = {m.I:.3f}   p-value = {m.p_sim:.3f}")

Hexagons with no neighbours: 0
has_bubble_tea        Moran's I = 0.137   p-value = 0.001
catchment_15_34       Moran's I = 0.866   p-value = 0.001
food_places_500m      Moran's I = 0.590   p-value = 0.001
